In [30]:
# Notebook ini adalah panduan praktik untuk Pertemuan 1.
# Jalankan cell-cell di bawah ini secara berurutan.

# Pertemuan 1: Pengantar Big Data dan Ekosistem Modern

## Tujuan Pembelajaran
1. Memahami konsep Big Data dan 5V
2. Mengenal ekosistem tools modern (DuckDB, Polars)
3. Membandingkan performa Pandas vs Polars
4. Memahami format data: CSV vs Parquet

---
## Bagian 1: Verifikasi Environment
Mari pastikan semua tools terinstal dengan benar.

In [31]:
import sys
print(f"Python version: {sys.version}")

# Core tools
import polars as pl
import duckdb
import plotly
import plotly.express as px
import pyarrow as pa

print(f"\nPolars version:  {pl.__version__}")
print(f"DuckDB version:  {duckdb.__version__}")
print(f"Plotly version:  {plotly.__version__}")
print(f"PyArrow version: {pa.__version__}")
print(f"\nSemua tools siap digunakan.")

Python version: 3.14.0 (tags/v3.14.0:ebf955d, Oct  7 2025, 10:15:03) [MSC v.1944 64 bit (AMD64)]

Polars version:  1.44.2
DuckDB version:  1.5.6
Plotly version:  7.1.0
PyArrow version: 25.0.1

Semua tools siap digunakan.


---
## Bagian 2: Hello World — DuckDB dan Polars

### 2a. DuckDB: SQL langsung di Python

In [32]:
# DuckDB: Buat query SQL langsung di Python
result = duckdb.sql("""
    SELECT
        'Hello Big Data!' AS pesan,
        42 AS angka_ajaib,
        CURRENT_TIMESTAMP AS waktu
""")
print(result)

┌─────────────────┬─────────────┬───────────────────────────────┐
│      pesan      │ angka_ajaib │             waktu             │
│     varchar     │    int32    │   timestamp with time zone    │
├─────────────────┼─────────────┼───────────────────────────────┤
│ Hello Big Data! │          42 │ 2026-10-01 13:16:37.075528+07 │
└─────────────────┴─────────────┴───────────────────────────────┘



In [33]:
# DuckDB bisa generate data untuk testing
duckdb.sql("""
    SELECT
        i AS id,
        'Mahasiswa_' || i AS nama,
        CASE WHEN i % 3 = 0 THEN 'Teknik Informatika'
             WHEN i % 3 = 1 THEN 'Sistem Informasi'
             ELSE 'Data Science'
        END AS jurusan,
        60 + (random() * 40)::INT AS nilai
    FROM generate_series(1, 10) AS t(i)
""").show()

┌───────┬──────────────┬────────────────────┬───────┐
│  id   │     nama     │      jurusan       │ nilai │
│ int64 │   varchar    │      varchar       │ int32 │
├───────┼──────────────┼────────────────────┼───────┤
│     1 │ Mahasiswa_1  │ Sistem Informasi   │    73 │
│     2 │ Mahasiswa_2  │ Data Science       │    99 │
│     3 │ Mahasiswa_3  │ Teknik Informatika │    79 │
│     4 │ Mahasiswa_4  │ Sistem Informasi   │    68 │
│     5 │ Mahasiswa_5  │ Data Science       │    67 │
│     6 │ Mahasiswa_6  │ Teknik Informatika │    78 │
│     7 │ Mahasiswa_7  │ Sistem Informasi   │    62 │
│     8 │ Mahasiswa_8  │ Data Science       │    81 │
│     9 │ Mahasiswa_9  │ Teknik Informatika │    92 │
│    10 │ Mahasiswa_10 │ Sistem Informasi   │    86 │
└───────┴──────────────┴────────────────────┴───────┘
  10 rows                                 4 columns



### 2b. Polars: DataFrame Modern

In [34]:
# Polars: Buat DataFrame
df = pl.DataFrame(
    {
        "nama": ["Andi", "Budi", "Citra", "Dewi", "Eka"],
        "kota": ["Jakarta", "Bandung", "Surabaya", "Jakarta", "Bandung"],
        "nilai": [85, 92, 78, 88, 95],
        "semester": [5, 5, 5, 7, 5],
    }
)
print(df)

shape: (5, 4)
┌───────┬──────────┬───────┬──────────┐
│ nama  ┆ kota     ┆ nilai ┆ semester │
│ ---   ┆ ---      ┆ ---   ┆ ---      │
│ str   ┆ str      ┆ i64   ┆ i64      │
╞═══════╪══════════╪═══════╪══════════╡
│ Andi  ┆ Jakarta  ┆ 85    ┆ 5        │
│ Budi  ┆ Bandung  ┆ 92    ┆ 5        │
│ Citra ┆ Surabaya ┆ 78    ┆ 5        │
│ Dewi  ┆ Jakarta  ┆ 88    ┆ 7        │
│ Eka   ┆ Bandung  ┆ 95    ┆ 5        │
└───────┴──────────┴───────┴──────────┘


In [35]:
# Polars expressions: filter, group, aggregate
hasil = (
    df
    .filter(pl.col("semester") == 5)
    .group_by("kota")
    .agg(
        pl.col("nilai").mean().alias("rata_rata_nilai"),
        pl.col("nama").count().alias("jumlah_mahasiswa"),
    )
    .sort("rata_rata_nilai", descending=True)
)
print(hasil)

shape: (3, 3)
┌──────────┬─────────────────┬──────────────────┐
│ kota     ┆ rata_rata_nilai ┆ jumlah_mahasiswa │
│ ---      ┆ ---             ┆ ---              │
│ str      ┆ f64             ┆ u32              │
╞══════════╪═════════════════╪══════════════════╡
│ Bandung  ┆ 93.5            ┆ 2                │
│ Jakarta  ┆ 85.0            ┆ 1                │
│ Surabaya ┆ 78.0            ┆ 1                │
└──────────┴─────────────────┴──────────────────┘


### 2c. Interoperabilitas DuckDB dan Polars

In [36]:
# DuckDB bisa query Polars DataFrame langsung
duckdb.sql("""
    SELECT kota, AVG(nilai) AS avg_nilai
    FROM df
    WHERE semester = 5
    GROUP BY kota
    ORDER BY avg_nilai DESC
""").show()

┌──────────┬───────────┐
│   kota   │ avg_nilai │
│ varchar  │  double   │
├──────────┼───────────┤
│ Bandung  │      93.5 │
│ Jakarta  │      85.0 │
│ Surabaya │      78.0 │
└──────────┴───────────┘



In [37]:
# Sebaliknya: DuckDB result ke Polars DataFrame
result_polars = duckdb.sql("""
    SELECT * FROM df WHERE nilai > 85
""").pl()

print(type(result_polars))
print(result_polars)

<class 'polars.dataframe.frame.DataFrame'>
shape: (3, 4)
┌──────┬─────────┬───────┬──────────┐
│ nama ┆ kota    ┆ nilai ┆ semester │
│ ---  ┆ ---     ┆ ---   ┆ ---      │
│ str  ┆ str     ┆ i64   ┆ i64      │
╞══════╪═════════╪═══════╪══════════╡
│ Budi ┆ Bandung ┆ 92    ┆ 5        │
│ Dewi ┆ Jakarta ┆ 88    ┆ 7        │
│ Eka  ┆ Bandung ┆ 95    ┆ 5        │
└──────┴─────────┴───────┴──────────┘


---
## Bagian 3: Benchmark — Pandas vs Polars

Mari kita buat dataset besar dan bandingkan performanya.

In [38]:
import time
import numpy as np
import pandas as pd

# Generate dataset 1 juta baris
N = 1_000_000
print(f"Generating {N:,} baris data...")

# Data untuk Pandas dan Polars
np.random.seed(42)
data = {
    "id": range(N),
    "kota": np.random.choice(
        ["Jakarta", "Bandung", "Surabaya", "Medan", "Semarang",
         "Makassar", "Palembang", "Yogyakarta", "Denpasar", "Manado"],
        N,
    ),
    "kategori": np.random.choice(
        ["Elektronik", "Fashion", "Makanan", "Kesehatan", "Olahraga"],
        N,
    ),
    "harga": np.random.uniform(10_000, 5_000_000, N).astype(int),
    "kuantitas": np.random.randint(1, 50, N),
}

Generating 1,000,000 baris data...


### 3a. Pandas

In [39]:
# Pandas
start = time.perf_counter()
df_pandas = pd.DataFrame(data)
result_pandas = (
    df_pandas
    .groupby(["kota", "kategori"])
    .agg(
        total_penjualan=("harga", "sum"),
        rata_rata_harga=("harga", "mean"),
        jumlah_transaksi=("id", "count"),
    )
    .reset_index()
    .sort_values("total_penjualan", ascending=False)
    .head(10)
)
waktu_pandas = time.perf_counter() - start
print(f"Pandas: {waktu_pandas:.4f} detik")
print(result_pandas)

Pandas: 0.4007 detik
          kota    kategori  total_penjualan  rata_rata_harga  jumlah_transaksi
33   Palembang     Makanan      50762394941     2.521603e+06             20131
41    Surabaya     Fashion      50707532785     2.502469e+06             20263
26       Medan     Fashion      50656184686     2.515827e+06             20135
0      Bandung  Elektronik      50635437819     2.526592e+06             20041
46  Yogyakarta     Fashion      50623174453     2.503124e+06             20224
38    Semarang     Makanan      50561727610     2.495643e+06             20260
37    Semarang   Kesehatan      50545092398     2.499386e+06             20223
12     Jakarta   Kesehatan      50491980622     2.501089e+06             20188
17    Makassar   Kesehatan      50490561824     2.519992e+06             20036
22      Manado   Kesehatan      50464873228     2.510066e+06             20105


### 3b. Polars

In [40]:
# Polars
start = time.perf_counter()
df_polars = pl.DataFrame(data)
result_polars = (
    df_polars
    .group_by(["kota", "kategori"])
    .agg(
        pl.col("harga").sum().alias("total_penjualan"),
        pl.col("harga").mean().alias("rata_rata_harga"),
        pl.col("id").count().alias("jumlah_transaksi"),
    )
    .sort("total_penjualan", descending=True)
    .head(10)
)
waktu_polars = time.perf_counter() - start
print(f"Polars: {waktu_polars:.4f} detik")
print(result_polars)

Polars: 0.4281 detik
shape: (10, 5)
┌────────────┬────────────┬─────────────────┬─────────────────┬──────────────────┐
│ kota       ┆ kategori   ┆ total_penjualan ┆ rata_rata_harga ┆ jumlah_transaksi │
│ ---        ┆ ---        ┆ ---             ┆ ---             ┆ ---              │
│ str        ┆ str        ┆ i64             ┆ f64             ┆ u32              │
╞════════════╪════════════╪═════════════════╪═════════════════╪══════════════════╡
│ Palembang  ┆ Makanan    ┆ 50762394941     ┆ 2.5216e6        ┆ 20131            │
│ Surabaya   ┆ Fashion    ┆ 50707532785     ┆ 2.5025e6        ┆ 20263            │
│ Medan      ┆ Fashion    ┆ 50656184686     ┆ 2.5158e6        ┆ 20135            │
│ Bandung    ┆ Elektronik ┆ 50635437819     ┆ 2.5266e6        ┆ 20041            │
│ Yogyakarta ┆ Fashion    ┆ 50623174453     ┆ 2.5031e6        ┆ 20224            │
│ Semarang   ┆ Makanan    ┆ 50561727610     ┆ 2.4956e6        ┆ 20260            │
│ Semarang   ┆ Kesehatan  ┆ 50545092398     ┆ 2.499

In [41]:
# Perbandingan
speedup = waktu_pandas / waktu_polars
print(f"\nHasil Benchmark ({N:,} baris):")
print(f"   Pandas: {waktu_pandas:.4f} detik")
print(f"   Polars: {waktu_polars:.4f} detik")
print(f"   Speedup: {speedup:.1f}x lebih cepat!")


Hasil Benchmark (1,000,000 baris):
   Pandas: 0.4007 detik
   Polars: 0.4281 detik
   Speedup: 0.9x lebih cepat!


---
## Bagian 4: Format Data — CSV vs Parquet

Parquet adalah format data kolumnar yang jauh lebih efisien daripada CSV.

In [42]:
from pathlib import Path

# Simpan sebagai CSV dan Parquet menggunakan folder lokal agar aman di Windows
base_dir = Path.cwd() / ".tmp_data"
base_dir.mkdir(exist_ok=True)
csv_path = base_dir / "demo_data.csv"
parquet_path = base_dir / "demo_data.parquet"

df_polars.write_csv(csv_path)
df_polars.write_parquet(parquet_path)

csv_size = csv_path.stat().st_size / (1024 * 1024)  # MB
parquet_size = parquet_path.stat().st_size / (1024 * 1024)  # MB

print(f"Ukuran file ({N:,} baris):")
print(f"   CSV:     {csv_size:.2f} MB")
print(f"   Parquet: {parquet_size:.2f} MB")
print(f"   Rasio:   Parquet {csv_size / parquet_size:.1f}x lebih kecil!")

Ukuran file (1,000,000 baris):
   CSV:     33.65 MB
   Parquet: 5.80 MB
   Rasio:   Parquet 5.8x lebih kecil!


In [43]:
# Benchmark: membaca CSV vs Parquet
start = time.perf_counter()
_ = pl.read_csv(csv_path)
waktu_csv = time.perf_counter() - start

start = time.perf_counter()
_ = pl.read_parquet(parquet_path)
waktu_parquet = time.perf_counter() - start

print(f"\nWaktu baca:")
print(f"   CSV:     {waktu_csv:.4f} detik")
print(f"   Parquet: {waktu_parquet:.4f} detik")
print(f"   Speedup: {waktu_csv / waktu_parquet:.1f}x lebih cepat!")


Waktu baca:
   CSV:     0.0300 detik
   Parquet: 0.0179 detik
   Speedup: 1.7x lebih cepat!


In [44]:
# DuckDB: Query langsung ke file Parquet (tanpa load ke memory)
duckdb.sql(f"""
    SELECT kota, COUNT(*) AS jumlah, AVG(harga) AS avg_harga
    FROM '{parquet_path}'
    GROUP BY kota
    ORDER BY jumlah DESC
""").show()

┌────────────┬────────┬────────────────────┐
│    kota    │ jumlah │     avg_harga      │
│  varchar   │ int64  │       double       │
├────────────┼────────┼────────────────────┤
│ Semarang   │ 100482 │ 2496291.1279731696 │
│ Surabaya   │ 100255 │ 2506790.9998703306 │
│ Jakarta    │ 100114 │ 2510696.0849231877 │
│ Bandung    │ 100049 │  2508726.660136533 │
│ Medan      │  99977 │  2496038.008712004 │
│ Makassar   │  99940 │ 2506172.1348709227 │
│ Manado     │  99882 │ 2505698.1008189665 │
│ Palembang  │  99839 │ 2507095.9009304983 │
│ Denpasar   │  99783 │ 2502364.6813385044 │
│ Yogyakarta │  99679 │ 2508790.1856057947 │
└────────────┴────────┴────────────────────┘
  10 rows                        3 columns



---
## Bagian 5: Visualisasi Pertama dengan Plotly

In [45]:
# Aggregate data untuk visualisasi
from pathlib import Path

viz_data = (
    df_polars
    .group_by("kota")
    .agg(
        pl.col("harga").sum().alias("total_penjualan"),
        pl.col("id").count().alias("jumlah_transaksi"),
    )
    .sort("total_penjualan", descending=True)
)

# Plotly bar chart
fig = px.bar(
    viz_data.to_pandas(),
    x="kota",
    y="total_penjualan",
    color="jumlah_transaksi",
    title="Total Penjualan per Kota (Data Simulasi)",
    labels={
        "kota": "Kota",
        "total_penjualan": "Total Penjualan (Rp)",
        "jumlah_transaksi": "Jumlah Transaksi",
    },
    color_continuous_scale="Viridis",
)
fig.update_layout(
    xaxis_tickangle=-45,
    template="plotly_dark",
)

# Simpan ke file HTML agar tidak bergantung pada renderer nbformat di VS Code
output_dir = Path.cwd() / ".tmp_data"
output_dir.mkdir(exist_ok=True)
plot_path = output_dir / "total_penjualan_per_kota.html"
fig.write_html(plot_path, include_plotlyjs="cdn")
print(f"Plot tersimpan di: {plot_path}")

Plot tersimpan di: d:\fix\materi-abd-2026\notebooks\P01_pengantar\.tmp_data\total_penjualan_per_kota.html


In [46]:
# Scatter plot: harga vs kuantitas (sample 10,000 dari 1 juta)
sample = df_polars.sample(n=10_000, seed=42)
fig2 = px.scatter(
    sample.to_pandas(),
    x="harga",
    y="kuantitas",
    color="kategori",
    title="Harga vs Kuantitas per Kategori (Sample 10K dari 1M)",
    labels={"harga": "Harga (Rp)", "kuantitas": "Kuantitas"},
    opacity=0.5,
    template="plotly_dark",
)

plot2_path = output_dir / "harga_vs_kuantitas.html"
fig2.write_html(plot2_path, include_plotlyjs="cdn")
print(f"Plot tersimpan di: {plot2_path}")

Plot tersimpan di: d:\fix\materi-abd-2026\notebooks\P01_pengantar\.tmp_data\harga_vs_kuantitas.html


---
## Ringkasan Pertemuan 1

Hari ini kita telah:
1. Setup environment (Docker + JupyterLab)
2. Berkenalan dengan DuckDB (SQL analytics) dan Polars (DataFrame modern)
3. Melihat interoperabilitas DuckDB dan Polars via Apache Arrow
4. Benchmark: Polars jauh lebih cepat dari Pandas
5. Format data: Parquet jauh lebih efisien dari CSV
6. Visualisasi pertama dengan Plotly

### Pertemuan Berikutnya
Polars Deep Dive — Lazy evaluation, expressions, window functions